# GOGET UK (NSTA/UKCS) Pipeline — Colab Edition

Automated ingestion pipeline for the UK GOGET cycle. Unlike the 2024 version of this
notebook, **none of the three NSTA source files need to be manually downloaded** —
this notebook fetches the production data, field-partner equity data, and field
outlines directly from NSTA's ArcGIS/Azure endpoints every time it runs, so it always
pulls the latest published data.

You only need to supply one file yourself: the current **GOGET UK field-level export**
(`goget-field-export-*.xlsx`, 3 sheets: main / reserves / production), since that comes
from GEM's own database export tool, not from NSTA.

**What this notebook does:**
1. Fetches NSTA production (PPRS), field-partner equity, and field-outline data live via API — no manual file wrangling.
2. Converts every quantity into GOGET's own unit conventions (confirmed against the live UK export: oil/condensate → `million bbl/y`, associated/dry gas → `million m³/y`).
3. Matches every NSTA field to its GOGET `Project ID` by normalized name (with a UKCS-licence-number cross-check).
4. Exports three timestamped files to Google Drive: production history, field attributes (incl. WKT outlines + current equity), and a match report flagging anything that needs human investigation.

Companion reference: `goget-uk-pipeline/SKILL.md` in the GOGET project folder documents the endpoints, unit conversions, and known name-matching edge cases (e.g. operator-suffixed duplicate field names) in more detail than the inline comments below.

In [37]:
# @title Step 0 — Install dependencies
!pip install -q shapely requests openpyxl fuzzywuzzy[speedup]
print("Ready.")

Ready.


## Step 1 — Configuration

In [38]:
# @title Configuration { display-mode: "form" }
MOUNT_DRIVE = True  # @param {type:"boolean"}

# Path to the GOGET Shared Drive UK folder (only used if MOUNT_DRIVE is True).
# Leave as-is unless GEM has moved the folder.
GOGET_DRIVE_FOLDER = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United Kingdom (GOGET O&G)/GOGET Exports/"  # @param {type:"string"}

# Filename of the current GOGET UK field-level export inside that folder
# (or a full path/Colab-upload — see Step 3). Leave blank to upload manually instead.
GOGET_EXPORT_FILENAME = "goget-field-export-2026-07-23T210040.xlsx"  # @param {type:"string"}

import datetime
RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Run timestamp: {RUN_TIMESTAMP}")

Run timestamp: 20260728_183105


In [39]:
# @title Mount Google Drive (skip if MOUNT_DRIVE is False)
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Skipping Drive mount — outputs will be written to /content/ only.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Fetch NSTA data live (no manual downloads)

All three NSTA datasets are hosted as public ArcGIS Hub items. Rather than clicking
through the Hub UI each cycle, we resolve each item's underlying service URL once and
query it directly. If NSTA ever republishes a service under a new item ID, re-resolve it
by fetching `https://open-data-ukcs-transition.hub.arcgis.com/datasets/UKCS-transition::<slug>/about`
and reading the `og:image` meta tag (it embeds the ArcGIS item ID) — see SKILL.md.

In [40]:
# @title Resolve ArcGIS item IDs -> service URLs
import requests

# Confirmed 2026-07 — re-derive per SKILL.md if NSTA republishes these services.
OUTLINES_ITEM_ID = "ba711a4be08449329a871d37ae910735"       # UKCS offshore hydrocarbon fields (WGS84)
PRODUCTION_ITEM_ID = "dd38204275a04618ab7ddd00f87224e3"      # UKCS hydrocarbon field production reports (PPRS points, WGS84)
FIELD_PARTNERS_ITEM_ID = "40fb75005dca48e886891350da9dedd8"  # Field Partners (document link)


def resolve_item(item_id):
    r = requests.get(f"https://www.arcgis.com/sharing/rest/content/items/{item_id}", params={"f": "json"}, timeout=30)
    r.raise_for_status()
    return r.json()


outlines_meta = resolve_item(OUTLINES_ITEM_ID)
production_meta = resolve_item(PRODUCTION_ITEM_ID)
partners_meta = resolve_item(FIELD_PARTNERS_ITEM_ID)

OUTLINES_SERVICE_URL = outlines_meta["url"]
PRODUCTION_SERVICE_URL = production_meta["url"]
FIELD_PARTNERS_DIRECT_URL = partners_meta["url"]  # direct .xlsx blob link

print("Outlines FeatureServer:   ", OUTLINES_SERVICE_URL)
print("Production FeatureServer: ", PRODUCTION_SERVICE_URL)
print("Field partners .xlsx:     ", FIELD_PARTNERS_DIRECT_URL)

Outlines FeatureServer:    https://services-eu1.arcgis.com/OZMfUznmLTnWccBc/arcgis/rest/services/Offshore_hydrocarbon_fields_(WGS84)/FeatureServer
Production FeatureServer:  https://services-eu1.arcgis.com/OZMfUznmLTnWccBc/arcgis/rest/services/UKCS_hydrocarbon_field_production_reports_PPRS_points_(WGS84)/FeatureServer
Field partners .xlsx:      https://datanstauthority.blob.core.windows.net/external/Documents/field_partners.xlsx


In [41]:
# @title Fetch field outlines (polygons + operator/licence/equity attributes)
import json

def fetch_layer_geojson(service_url, layer=0, out_sr=4326):
    url = f"{service_url}/{layer}/query"
    params = {"where": "1=1", "outFields": "*", "f": "geojson", "outSR": out_sr}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return r.json()

outlines_geojson = fetch_layer_geojson(OUTLINES_SERVICE_URL)
print(f"Fetched {len(outlines_geojson['features'])} field outlines")

Fetched 478 field outlines


In [42]:
import time

def fetch_layer_records_paginated(service_url, layer=0, page_size=2000, sleep_s=0.1):
    """ArcGIS Feature Services cap query results (commonly 2000/page) - page through
    with resultOffset until a page comes back short."""
    url = f"{service_url}/{layer}/query"
    all_features = []
    offset = 0
    while True:
        params = {
            "where": "1=1", "outFields": "*", "f": "json",
            "returnGeometry": "false",
            "resultRecordCount": page_size, "resultOffset": offset,
        }
        r = requests.get(url, params=params, timeout=120)
        r.raise_for_status()
        data = r.json()
        feats = data.get("features", [])
        all_features.extend(feats)
        if offset % (page_size * 10) == 0:
            print(f"  ... {len(all_features)} rows fetched")
        if len(feats) < page_size:
            break
        offset += page_size
        time.sleep(sleep_s)
    return data.get("fields", []), all_features


prod_fields, prod_features = fetch_layer_records_paginated(PRODUCTION_SERVICE_URL)
alias_map = {f["name"]: f["alias"] for f in prod_fields}  # auto column renaming - no PPRS_Column_Names.xlsx needed
production_raw = pd.DataFrame([f["attributes"] for f in prod_features]) if False else None
import pandas as pd
production_raw = pd.DataFrame([f["attributes"] for f in prod_features])
production_raw = production_raw.rename(columns=alias_map)
print(f"Fetched {len(production_raw)} production rows, {production_raw.shape[1]} columns")
production_raw.head()

  ... 2000 rows fetched
  ... 22000 rows fetched
  ... 42000 rows fetched
  ... 62000 rows fetched
  ... 82000 rows fetched
  ... 102000 rows fetched
  ... 122000 rows fetched
Fetched 133702 production rows, 46 columns


,OBJECTID,Field name,Field region,"Onshore, offshore flag",Operator organisation group,Reporting unit name,Reporting unit type code,Reporting unit type description,Reporting period in yyyymm format,Reporting period year,...,Gas flared - density (kg/sm3),Gas flared - non hydrocarbon (%),Gas vented - volume (ksm3),Gas vented - density (kg/sm3),Gas vented - non hydrocarbon (%),Injected water - volume (m3),Injected water - volume (mb/d),Produced water - volume (m3),Produced water - volume (mb/d),Reinjected produced water - volume (m3)
0,1,ABIGAIL,CNS,Offshore,ITHACA ENERGY,ABIGAIL,P,Oil Field Exporting to Pipeline,202210,2022,...,1.321,3.02,5.0,1.368,3.02,0.0,0.0,1393.0,0.282644,0.0
1,2,ABIGAIL,CNS,Offshore,ITHACA ENERGY,ABIGAIL,P,Oil Field Exporting to Pipeline,202211,2022,...,0.920,3.02,12.0,1.368,3.02,0.0,0.0,2599.0,0.544924,0.0
2,3,ABIGAIL,CNS,Offshore,ITHACA ENERGY,ABIGAIL,P,Oil Field Exporting to Pipeline,202212,2022,...,1.189,3.02,12.0,1.368,3.02,0.0,0.0,2244.0,0.455315,0.0
3,4,ABIGAIL,CNS,Offshore,ITHACA ENERGY,ABIGAIL,P,Oil Field Exporting to Pipeline,202301,2023,...,1.125,3.02,0.0,0.000,3.02,0.0,0.0,373.0,0.075683,0.0
4,5,ABIGAIL,CNS,Offshore,ITHACA ENERGY,ABIGAIL,P,Oil Field Exporting to Pipeline,202302,2023,...,1.076,3.02,0.0,0.000,3.02,0.0,0.0,277.0,0.062226,0.0


In [43]:
# @title Fetch field partners (equity history) — direct download, no Hub UI needed
import io

resp = requests.get(FIELD_PARTNERS_DIRECT_URL, timeout=60)
resp.raise_for_status()
field_partners_raw = pd.read_excel(io.BytesIO(resp.content))
print(f"Fetched {len(field_partners_raw)} field-partner equity rows, {field_partners_raw['Field Name'].nunique()} unique fields")
field_partners_raw.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Fetched 7628 field-partner equity rows, 531 unique fields


,Field Name,On Offshore,Median Line Flag,Status,Organisation Name,Percentage Holding,Operator Flag,Start Date,End Date,Equity Share Time Period
0,ABIGAIL,Offshore,NaN,700 - PRODUCING,ITHACA ENERGY DEVELOPMENTS UK LIMITED,20.0,N,2020-10-15,2021-11-30,Previous-2020 to 2021
1,ABIGAIL,Offshore,NaN,700 - PRODUCING,ITHACA ENERGY (UK) LIMITED,80.0,Y,2020-10-15,2021-11-30,Previous-2020 to 2021
2,ABIGAIL,Offshore,NaN,700 - PRODUCING,ITHACA ENERGY DEVELOPMENTS UK LIMITED,20.0,N,2021-11-30,2023-12-21,Previous-2021 to 2023
3,ABIGAIL,Offshore,NaN,700 - PRODUCING,ITHACA ENERGY (UK) LIMITED,80.0,Y,2021-11-30,2023-12-21,Previous-2021 to 2023
4,ABIGAIL,Offshore,NaN,700 - PRODUCING,ITHACA ENERGY DEVELOPMENTS UK LIMITED,20.0,N,2023-12-21,2024-05-30,Previous-2023 to 2024


## Step 3 — Load the GOGET UK field export

This is the one file this notebook can't fetch automatically (it comes from GEM's
internal database export tool). Either:
- set `GOGET_EXPORT_FILENAME` in Step 1 to a file already sitting in `GOGET_DRIVE_FOLDER`, or
- leave it blank and use the upload widget below.

In [44]:
# @title Load or upload the GOGET export
import os

goget_path = None
if GOGET_EXPORT_FILENAME:
    candidate = os.path.join(GOGET_DRIVE_FOLDER, GOGET_EXPORT_FILENAME)
    if os.path.exists(candidate):
        goget_path = candidate

if goget_path is None:
    print("Please upload the goget-field-export-*.xlsx file:")
    from google.colab import files
    uploaded = files.upload()
    goget_path = next(iter(uploaded.keys()))

print(f"Using GOGET export: {goget_path}")
goget_main_sheet = pd.read_excel(goget_path, sheet_name="Field-level main data")
print(f"{len(goget_main_sheet)} GOGET UK field rows loaded")

Using GOGET export: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United Kingdom (GOGET O&G)/GOGET Exports/goget-field-export-2026-07-23T210040.xlsx
425 GOGET UK field rows loaded


## Step 4 — Pipeline functions (name normalization, unit conversion, matching)

In [45]:
import re
import numpy as np

# Confirmed against the live GOGET UK export: oil & gas condensate -> million bbl/y;
# associated gas & dry gas -> million m3/y. Shared boe basis matches the GOGET brief's
# own gas->boe factor for cross-checks against external benchmarks (0.00588 boe/m3).
M3_TO_BBL = 6.2898
GAS_M3_TO_BOE = 0.00588
OIL_BBL_TO_BOE = 1.0

FUEL_COLUMN_MAP = {
    "Oil production - volume (m3)": ("oil", "m3"),
    "Gas condensate - volume (m3)": ("gas condensate", "m3"),
    "Associated gas production - volume (ksm3)": ("associated gas", "ksm3"),
    "Dry gas production - volume (ksm3)": ("dry gas", "ksm3"),
}


def normalize_field_name(name):
    """Fold GOGET's composed names ('X Oil and Gas Field (United Kingdom)') and NSTA's
    bare names ('X' / 'X FIELD') down to a common key for matching."""
    if not isinstance(name, str):
        return ""
    n = name.strip().upper()
    n = re.sub(r"\s*\([^)]*UNITED KINGDOM\)\s*$", "", n)
    n = re.sub(r"\s+(OIL AND GAS FIELD|OIL FIELD|GAS FIELD|OIL AND GAS ASSET|OIL ASSET|GAS ASSET)$", "", n)
    n = re.sub(r"\s+FIELD$", "", n)
    n = n.replace("-", " ").replace("'", "").replace(".", "")
    n = re.sub(r"\s+", " ", n).strip()
    return n


def extract_ukcs_licences(gov_unit_id):
    """Pull UKCS licence numbers + the legacy Historical GOGET ID out of GOGET's
    composite 'Government unit ID' cell, e.g. 'UKCS: P1665, Historical GOGET: OG0018223'."""
    if not isinstance(gov_unit_id, str):
        return [], None
    licences = re.findall(r"UKCS:\s*([A-Z]{1,3}\d+[A-Za-z]?)", gov_unit_id)
    hist = re.search(r"Historical GOGET:\s*(OG\d+)", gov_unit_id)
    return licences, (hist.group(1) if hist else None)

print("Helpers defined.")

Helpers defined.


In [46]:
# @title Process production data -> annual, GOGET-unit-converted
colmap_lower = {c.lower(): c for c in production_raw.columns}

def col(name):
    return colmap_lower.get(name.lower())

field_col = col("Field name")
year_col = col("Reporting period year")
onoff_col = col("Onshore, offshore flag")

rows = []
for nsta_col, (fuel, unit) in FUEL_COLUMN_MAP.items():
    actual = col(nsta_col)
    if actual is None:
        print(f"  [warn] column not found: {nsta_col}")
        continue
    sub = production_raw[[field_col, year_col, onoff_col, actual]].copy()
    sub.columns = ["field_name", "year", "onshore_offshore", "native_qty"]
    sub["fuel"] = fuel
    sub["native_unit"] = unit
    rows.append(sub)

production_long = pd.concat(rows, ignore_index=True)
production_long["native_qty"] = pd.to_numeric(production_long["native_qty"], errors="coerce").fillna(0.0)
production_long["norm_name"] = production_long["field_name"].apply(normalize_field_name)

production_annual = (
    production_long.groupby(["norm_name", "field_name", "year", "fuel", "native_unit", "onshore_offshore"],
                             as_index=False)["native_qty"].sum()
)

def convert(row):
    if row["native_unit"] == "m3":
        return row["native_qty"] * M3_TO_BBL / 1e6, "million bbl/y"
    if row["native_unit"] == "ksm3":
        return row["native_qty"] / 1000.0, "million m3/y"
    return np.nan, None

converted = production_annual.apply(convert, axis=1, result_type="expand")
production_annual["quantity_converted"] = converted[0]
production_annual["units_converted"] = converted[1]
production_annual["quantity_boe_millions"] = np.where(
    production_annual["units_converted"] == "million bbl/y",
    production_annual["quantity_converted"] * OIL_BBL_TO_BOE,
    production_annual["quantity_converted"] * GAS_M3_TO_BOE,
)
production_annual = production_annual.sort_values(["norm_name", "year", "fuel"]).reset_index(drop=True)
print(f"{len(production_annual)} field-year-fuel rows, {production_annual['norm_name'].nunique()} unique fields, "
      f"years {int(production_annual['year'].min())}-{int(production_annual['year'].max())}")
production_annual.head()

46808 field-year-fuel rows, 551 unique fields, years 1975-2026


,norm_name,field_name,year,fuel,native_unit,onshore_offshore,native_qty,quantity_converted,units_converted,quantity_boe_millions
0,ABIGAIL,ABIGAIL,2022,associated gas,ksm3,Offshore,15170.0,15.170000,million m3/y,0.089200
1,ABIGAIL,ABIGAIL,2022,dry gas,ksm3,Offshore,0.0,0.000000,million m3/y,0.000000
2,ABIGAIL,ABIGAIL,2022,gas condensate,m3,Offshore,0.0,0.000000,million bbl/y,0.000000
3,ABIGAIL,ABIGAIL,2022,oil,m3,Offshore,50905.0,0.320182,million bbl/y,0.320182
4,ABIGAIL,ABIGAIL,2023,associated gas,ksm3,Offshore,50808.0,50.808000,million m3/y,0.298751


In [47]:
# @title Process field partners -> current ownership summary
field_partners_raw["norm_name"] = field_partners_raw["Field Name"].apply(normalize_field_name)

is_current = field_partners_raw["Equity Share Time Period"].astype(str).str.strip().str.lower() == "current"
current_partners = field_partners_raw[is_current].copy()
have_current = set(current_partners["norm_name"])
missing = field_partners_raw[~field_partners_raw["norm_name"].isin(have_current)].copy()
if not missing.empty:
    missing["End Date"] = pd.to_datetime(missing["End Date"], errors="coerce")
    idx = missing.groupby("norm_name")["End Date"].idxmax().dropna()
    current_partners = pd.concat([current_partners, missing.loc[idx]], ignore_index=True)


def _agg_ownership(g):
    g = g.sort_values("Percentage Holding", ascending=False)
    owners = "; ".join(f"{r['Organisation Name']} ({r['Percentage Holding']:.1f}%)" for _, r in g.iterrows())
    op_rows = g[g["Operator Flag"].astype(str).str.upper().isin(["Y", "YES", "TRUE", "1"])]
    operator = op_rows.iloc[0]["Organisation Name"] if not op_rows.empty else g.iloc[0]["Organisation Name"]
    return pd.Series({
        "current_owners": owners,
        "current_operator_from_partners": operator,
        "status_code_from_partners": g.iloc[0].get("Status", np.nan),
    })

ownership_summary = (
    current_partners.groupby("norm_name")[current_partners.columns.difference(["norm_name"])]
    .apply(_agg_ownership).reset_index()
)
print(f"{len(ownership_summary)} fields with a current-ownership summary")
ownership_summary.head()

531 fields with a current-ownership summary


,norm_name,current_owners,current_operator_from_partners,status_code_from_partners
0,ABIGAIL,ITHACA SP E&P LIMITED (100.0%),ITHACA SP E&P LIMITED,700 - PRODUCING
1,AFFLECK,NEO NEXT+ ENERGY PUK LIMITED (100.0%),NEO NEXT+ ENERGY PUK LIMITED,700 - PRODUCING
2,AIRTH COAL BED METHANE DEVT,INEOS UPSTREAM LIMITED (100.0%),INEOS UPSTREAM LIMITED,900 - PRODUCTION CEASED
3,ALBA,ITHACA OIL AND GAS LIMITED (36.7%); WALDORF PR...,ITHACA OIL AND GAS LIMITED,900 - PRODUCTION CEASED
4,ALBURY,ISLAND GAS LIMITED (100.0%),ISLAND GAS LIMITED,700 - PRODUCING


In [48]:
# @title Process field outlines -> attributes table (status, dates, licences, centroid, WKT)
from shapely.geometry import shape
from shapely import wkt as shapely_wkt

records = []
for feat in outlines_geojson["features"]:
    props = feat["properties"]
    geom = feat.get("geometry")
    centroid_lon = centroid_lat = None
    wkt_str = None
    if geom:
        try:
            g = shape(geom)
            c = g.centroid
            centroid_lon, centroid_lat = round(c.x, 5), round(c.y, 5)
            simplified = g.simplify(0.0001, preserve_topology=True)
            wkt_str = shapely_wkt.dumps(simplified, rounding_precision=5)  # keeps Excel cells well under the 32,767-char limit
        except Exception as e:
            print(f"  [warn] geometry failure for {props.get('FIELDNAME')}: {e}")

    # NSTA's outline attrs number the 2nd-20th equity holder (CURRLIC2/EQUITY2 ...
    # CURRLIC20/EQUITY20) but the *first* holder's fields are unnumbered
    # ("CURRLIC" / "EQUITY") - include that slot explicitly or it's silently dropped.
    lic_slots = ["CURRLIC"] + [f"CURRLIC{i}" for i in range(2, 21)]
    eq_slots = ["EQUITY"] + [f"EQUITY{i}" for i in range(2, 21)]
    currlic = [props.get(k) for k in lic_slots if props.get(k)]
    equity = [props.get(k) for k, lic in zip(eq_slots, lic_slots) if props.get(lic)]

    records.append({
        "norm_name": normalize_field_name(props.get("FIELDNAME", "")),
        "nsta_field_name": props.get("FIELDNAME"),
        "fieldtype": props.get("FIELDTYPE"),
        "status_outline": props.get("STATUS"),
        "det_status": props.get("DET_STATUS"),
        "discovery_date": props.get("DISC_DATE"),
        "discovery_well": props.get("DISC_WELL"),
        "production_start_date": props.get("PROD_DATE"),
        "water_depth_m": props.get("DEPTH_M"),
        "original_operator": props.get("ORIG_OP"),
        "original_licence": props.get("ORIG_LIC"),
        "current_operator": props.get("CURR_OPER"),
        "current_licences": "; ".join(currlic) if currlic else None,
        "current_equity_from_outline": ", ".join(f"{lic}:{eq}%" for lic, eq in zip(currlic, equity)) if currlic else None,
        "centroid_lon": centroid_lon,
        "centroid_lat": centroid_lat,
        "outline_wkt": wkt_str,
    })

outline_attrs = pd.DataFrame(records)
print(f"{len(outline_attrs)} field outlines ({outline_attrs['outline_wkt'].notna().sum()} with usable geometry)")
outline_attrs.head()

478 field outlines (478 with usable geometry)


,norm_name,nsta_field_name,fieldtype,status_outline,det_status,discovery_date,discovery_well,production_start_date,water_depth_m,original_operator,original_licence,current_operator,current_licences,current_equity_from_outline,centroid_lon,centroid_lat,outline_wkt
0,AFFLECK,AFFLECK,OIL,PRODUCING,DETERMINED,1975/04,30/19- 2,2009/08,73.76,MAERSK,P255,NEO NEXT+ ENERGY,NEO NEXT+ ENERGY PUK LIMITED,NEO NEXT+ ENERGY PUK LIMITED:100%,2.73247,56.41833,"POLYGON ((2.71927 56.40096, 2.71406 56.40442, ..."
1,ALBA,ALBA,OIL,PRODUCTION CEASED,DETERMINED,1984/12,16/26- 5,1994/01,140.21,CHEVRON,P213,ITHACA ENERGY,ENQUEST HEATHER LIMITED; EQUINOR UK LIMITED; I...,"ENQUEST HEATHER LIMITED:8%, EQUINOR UK LIMITED...",1.10846,58.03553,"POLYGON ((1.11305 58.01970, 1.11101 58.02491, ..."
2,ALDER,ALDER,COND,PRODUCTION CEASED,DETERMINED,1975/10,15/29- 2A,2016/11,154.84,CHEVRON,P119,ITHACA ENERGY,CHRYSAOR PRODUCTION (U.K.) LIMITED; ITHACA ENE...,"CHRYSAOR PRODUCTION (U.K.) LIMITED:26%, ITHACA...",0.69935,58.09955,"POLYGON ((0.62458 58.10202, 0.63626 58.10096, ..."
3,ALMA,ALMA,OIL,PRODUCTION CEASED,DETERMINED,1971/08,30/24- 2,2015/10,78,ENQUEST,P1765,ENQUEST PLC,ENQUEST HEATHER LIMITED; KUFPEC UK LIMITED,"ENQUEST HEATHER LIMITED:65%, KUFPEC UK LIMITED...",2.77946,56.18587,"POLYGON ((2.76461 56.19627, 2.76538 56.19743, ..."
4,ANDREW,ANDREW,OIL,PRODUCING,DETERMINED,1974/06,16/28- 1,1996/06,115.8,BP,P092,BP EXPLORATION,BP EXPLORATION OPERATING COMPANY LIMITED; NEO ...,"BP EXPLORATION OPERATING COMPANY LIMITED:90%, ...",1.40926,58.04099,"POLYGON ((1.41703 58.06099, 1.41494 58.05937, ..."


In [49]:
# @title Build the GOGET crosswalk + match report
goget_main_sheet["norm_name"] = goget_main_sheet["Unit Name"].apply(normalize_field_name)
_lic_hist = goget_main_sheet["Government unit ID"].apply(extract_ukcs_licences)
goget_main_sheet["ukcs_licences"] = _lic_hist.apply(lambda t: t[0])
goget_main_sheet["historical_goget_id"] = _lic_hist.apply(lambda t: t[1])

nsta_names = set(production_annual["norm_name"]) | set(outline_attrs["norm_name"])
goget_names = set(goget_main_sheet["norm_name"])
matched = goget_names & nsta_names
goget_only = goget_names - nsta_names
nsta_only = nsta_names - goget_names

match_rows = (
    [{"norm_name": n, "match_status": "matched"} for n in sorted(matched)]
    + [{"norm_name": n, "match_status": "goget_only_needs_investigation"} for n in sorted(goget_only)]
    + [{"norm_name": n, "match_status": "nsta_only_candidate_new_or_rename"} for n in sorted(nsta_only)]
)
match_report = pd.DataFrame(match_rows)
print(f"{len(matched)} matched, {len(goget_only)} GOGET-only, {len(nsta_only)} NSTA-only "
      f"({len(matched) / len(goget_names):.0%} of the GOGET UK roster auto-matched)")

375 matched, 50 GOGET-only, 221 NSTA-only (88% of the GOGET UK roster auto-matched)


In [53]:
# @title Assemble final field-attributes table
field_attributes = outline_attrs.merge(ownership_summary, on="norm_name", how="outer")
field_attributes = field_attributes.merge(
    goget_main_sheet[["norm_name", "Project ID", "Unit Name", "ukcs_licences", "historical_goget_id",
                       "Status", "Operator", "Latitude", "Longitude", "Discovery year"]].rename(columns={
                           "Project ID": "goget_project_id",
                           "Unit Name": "goget_unit_name",
                           "Latitude": "goget_latitude",
                           "Longitude": "goget_longitude",
                           "Status": "goget_status",
                           "Discovery year": "goget_discovery_year",
                           "Operator": "goget_operator",
                       }),
    on="norm_name", how="left",
)

production_annual_joined = production_annual.merge(
    goget_main_sheet[["norm_name", "Project ID", "Unit Name"]].rename(
        columns={"Project ID": "goget_project_id", "Unit Name": "goget_unit_name"}),
    on="norm_name", how="left",
)
print("tables assembled:", field_attributes.shape, production_annual_joined.shape, match_report.shape)

tables assembled: (602, 29) (46808, 12) (646, 2)


Step 5 - Automated Comparison

In [55]:
# @title Automated Comparison
# compare operators in field_attributes
from fuzzywuzzy import fuzz
import pandas as pd

def fuzzy_match_ratio(s1, s2):
    if pd.isna(s1) or pd.isna(s2):
        return 0
    return fuzz.ratio(str(s1), str(s2))

field_attributes['operator_fuzzy_match_ratio'] = field_attributes.apply(
    lambda row: fuzzy_match_ratio(row['current_operator'], row['goget_operator']),
    axis=1
)

print("Operator comparison using fuzzy matching (top 10 by lowest match ratio): ")
field_attributes_sorted_by_fuzzy = field_attributes.sort_values(by='operator_fuzzy_match_ratio', ascending=True)
display(field_attributes_sorted_by_fuzzy[['nsta_field_name', 'current_operator', 'goget_operator', 'operator_fuzzy_match_ratio']].head(10))

# compare locations between datasets
from geopy.distance import geodesic
import numpy as np

def calculate_distance(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan
    return geodesic((lat1, lon1), (lat2, lon2)).kilometers

field_attributes['distance_to_goget'] = field_attributes.apply(
    lambda row: calculate_distance(row['centroid_lat'], row['centroid_lon'], row['goget_latitude'], row['goget_longitude']),
    axis=1
)

print("Distance comparison (top 10 by lowest distance): ")
field_attributes_sorted_by_distance = field_attributes.sort_values(by='distance_to_goget', ascending=True)
display(field_attributes_sorted_by_distance[['nsta_field_name', 'centroid_lat', 'centroid_lon', 'goget_latitude', 'goget_longitude', 'distance_to_goget']].head(10))

# status comparison, define mapping NSTA "PRODUCING" = GOGET "Operating", NSTA "PRODUCTION CEASED" = GOGET "Mothballed"
# Invert the mapping to correctly convert GOGET status to NSTA status for comparison
inverted_status_mapping = {
    'Operating': 'PRODUCING',
    'Mothballed': 'PRODUCTION CEASED'
}

# Convert goget_status to title case to ensure consistent matching with the mapping
field_attributes['goget_status_processed'] = field_attributes['goget_status'].astype(str).str.title()
field_attributes['goget_status_mapped'] = field_attributes['goget_status_processed'].map(inverted_status_mapping)
field_attributes['status_comparison'] = np.where(
    field_attributes['status_outline'] == field_attributes['goget_status_mapped'],
    'Match',
    'No Match'
)

print("Status comparison added")

print("Status comparison (top 10 by status_comparison and nsta_field_name): ")
field_attributes_sorted_by_status = field_attributes.sort_values(by=['status_comparison', 'nsta_field_name'], ascending=[True, True])
display(field_attributes_sorted_by_status[['nsta_field_name', 'status_outline', 'goget_status', 'goget_status_processed', 'goget_status_mapped', 'status_comparison']].head(10))

# compare discovery year
# clean discovery_date - strip months YYYY/MM
field_attributes['discovery_date_cleaned'] = field_attributes['discovery_date'].str.split('/').str[0]
field_attributes['discovery_year_cleaned'] = field_attributes['discovery_date_cleaned'].str.strip()
field_attributes['discovery_year_cleaned'] = pd.to_numeric(field_attributes['discovery_year_cleaned'], errors='coerce')

# calculate difference from discovery_year_cleaned and goget_discovery_year
field_attributes['discovery_year_difference'] = field_attributes['discovery_year_cleaned'] - field_attributes['goget_discovery_year']

# flag difference greater than 1 year
field_attributes['discovery_year_difference_flag'] = np.where(
    field_attributes['discovery_year_difference'] > 1,
    'Yes',
    'No'
)

print("Discovery year comparison (top 10 by discovery_year_difference_flag): ")
field_attributes_sorted_by_discovery_year = field_attributes.sort_values(by='discovery_year_difference_flag', ascending=True)
display(field_attributes_sorted_by_discovery_year[['nsta_field_name', 'discovery_year_cleaned', 'goget_discovery_year', 'discovery_year_difference', 'discovery_year_difference_flag']].head(10))

Operator comparison using fuzzy matching (top 10 by lowest match ratio): 


,nsta_field_name,current_operator,goget_operator,operator_fuzzy_match_ratio
351,NaN,NaN,NaN,0
453,NaN,NaN,NaN,0
226,NaN,NaN,NaN,0
225,FLORA,HESS LIMITED,NaN,0
254,GAWAIN,PERENCO OIL & GAS,NaN,0
251,GANYMEDE,HARBOUR ENERGY PLC,NaN,0
239,NaN,NaN,Star Energy Group,0
237,FULMAR,NEO NEXT+ ENERGY,NaN,0
236,FRIGG,NEO NEXT+ ENERGY,NaN,0
235,NaN,NaN,NaN,0


Distance comparison (top 10 by lowest distance): 


,nsta_field_name,centroid_lat,centroid_lon,goget_latitude,goget_longitude,distance_to_goget
78,BOYLE,53.0254,2.89370,53.0254,2.8937,0.000000
123,CARNOUSTIE,57.3799,1.36590,57.3799,1.3659,0.000000
504,SOUTH SEAN,53.1868,2.82660,53.1868,2.8266,0.000000
257,GLENELG,56.9882,1.78950,56.9882,1.7895,0.000000
294,HIGHLANDER,58.4017,-0.14950,58.4017,-0.1495,0.000000
238,GADWALL,57.4296,0.70960,57.4296,0.7096,0.000000
340,KRAKEN,59.9091,1.25030,59.9091,1.2503,0.000000
197,EDRADOUR,60.9337,-2.22771,60.9337,-2.2277,0.000542
445,PETRONELLA,58.3387,-0.10461,58.3387,-0.1046,0.000586
390,MAULE,57.7286,0.98641,57.7286,0.9864,0.000596


Status comparison added
Status comparison (top 10 by status_comparison and nsta_field_name): 


,nsta_field_name,status_outline,goget_status,goget_status_processed,goget_status_mapped,status_comparison
0,ABIGAIL,PRODUCING,operating,Operating,PRODUCING,Match
9,ALLIGIN,PRODUCING,operating,Operating,PRODUCING,Match
10,ALMA,PRODUCTION CEASED,mothballed,Mothballed,PRODUCTION CEASED,Match
13,ALWYN NORTH,PRODUCING,operating,Operating,PRODUCING,Match
14,AMETHYST EAST,PRODUCTION CEASED,mothballed,Mothballed,PRODUCTION CEASED,Match
15,AMETHYST WEST,PRODUCTION CEASED,mothballed,Mothballed,PRODUCTION CEASED,Match
16,ANDREW,PRODUCING,operating,Operating,PRODUCING,Match
20,APOLLO,PRODUCING,operating,Operating,PRODUCING,Match
21,ARBROATH,PRODUCING,operating,Operating,PRODUCING,Match
22,ARKWRIGHT,PRODUCING,operating,Operating,PRODUCING,Match


Discovery year comparison (top 10 by discovery_year_difference_flag): 


,nsta_field_name,discovery_year_cleaned,goget_discovery_year,discovery_year_difference,discovery_year_difference_flag
0,ABIGAIL,NaN,NaN,NaN,No
397,MILLER,1983.0,NaN,NaN,No
398,MILLOM,1982.0,1982.0,0.0,No
399,MIMAS,1989.0,NaN,NaN,No
400,MINERVA,1969.0,1969.0,0.0,No
401,MINKE,1993.0,NaN,NaN,No
402,MIRREN,1993.0,1993.0,0.0,No
396,MERLIN,1997.0,NaN,NaN,No
403,MOIRA,1988.0,NaN,NaN,No
405,MONTROSE,1971.0,1971.0,0.0,No


## Step 6 — Export

In [56]:
# @title Export timestamped outputs
out_dir = GOGET_DRIVE_FOLDER if MOUNT_DRIVE else "/content/"

prod_path = f"{out_dir}NSTA_production_annual_{RUN_TIMESTAMP}.csv"
attrs_path = f"{out_dir}NSTA_field_attributes_{RUN_TIMESTAMP}.csv"
match_path = f"{out_dir}NSTA_match_report_{RUN_TIMESTAMP}.csv"

production_annual_joined.to_csv(prod_path, index=False)
field_attributes.to_csv(attrs_path, index=False)
match_report.to_csv(match_path, index=False)

print("Exported:")
print(" ", prod_path)
print(" ", attrs_path)
print(" ", match_path)

Exported:
  /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United Kingdom (GOGET O&G)/GOGET Exports/NSTA_production_annual_20260728_183105.csv
  /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United Kingdom (GOGET O&G)/GOGET Exports/NSTA_field_attributes_20260728_183105.csv
  /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/United Kingdom (GOGET O&G)/GOGET Exports/NSTA_match_report_20260728_183105.csv


## Notes

- **Match rate context:** ~88% auto-match was typical as of the 2026-07 cycle. The
  remainder splits into two known patterns worth checking by hand rather than forcing
  a match: (1) large discoveries not yet "field-determined" by NSTA (so absent from the
  outlines/production feeds) — e.g. Cambo-adjacent fields; (2) operator-suffixed
  duplicate NSTA names for the same physical field across an operatorship change (e.g.
  `BELL` / `BELL [CONOCOPHILLIPS]` / `BELL [PERENCO]` vs GOGET's single `Bell (Perenco)`
  row) — the same pattern the Netherlands cycle hit with K12-B9 Eni/Tenaz.
- **Outline equity data is more current than the field-partners file** for present-day
  ownership — use `current_equity_from_outline` as primary, `current_owners` (from
  field_partners.xlsx) mainly for historical equity-change verification (Phase 3a/3b).
- Re-derive the ArcGIS item IDs in Step 2 if NSTA republishes a service (rare, but
  update `goget-uk-pipeline/SKILL.md` in the GOGET project folder if it happens).